In [35]:
import pandas as pd
import plotly.express as px

In [36]:
state_codes = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "District of Columbia": "DC",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
}

In [37]:
def plot_usa_heatmap(
    df: pd.DataFrame,
    category_col: str,
    color_continuous_scale: str,
    title: str,
    hover_data: list[str] = None,
) -> None:
    fig = px.choropleth(
        df,
        locations="code",
        locationmode="USA-states",
        scope="usa",
        hover_name="state",
        hover_data=hover_data,
        color=category_col,
        color_continuous_scale=color_continuous_scale,
        title=title,
    )
    fig.update_layout(title_x=0.5, margin=dict(l=0, r=0, t=60, b=0))
    fig.show()

In [38]:
df = pd.read_parquet("../data/incidents.parquet")
df["code"] = df["state"].map(state_codes)
df.head()

,date,state,city_or_county,address,n_killed,n_injured,incident_url,source_url,incident_url_fields_missing,congressional_district,...,participant_name,participant_relationship,participant_status,participant_type,sources,state_house_district,state_senate_district,year,month,code
incident_id,,,,,,,,,,,,,,,,,,,,,
461105,2013-01-01,Pennsylvania,Mckeesport,1506 Versailles Avenue and Coursin Street,0,4,http://www.gunviolencearchive.org/incident/461105,http://www.post-gazette.com/local/south/2013/0...,False,14.0,...,0::Julian Sims,NaN,0::Arrested||1::Injured||2::Injured||3::Injure...,0::Victim||1::Victim||2::Victim||3::Victim||4:...,http://pittsburgh.cbslocal.com/2013/01/01/4-pe...,NaN,NaN,2013,1,PA
460726,2013-01-01,California,Hawthorne,13500 block of Cerise Avenue,1,3,http://www.gunviolencearchive.org/incident/460726,http://www.dailybulletin.com/article/zz/201301...,False,43.0,...,0::Bernard Gillis,NaN,0::Killed||1::Injured||2::Injured||3::Injured,0::Victim||1::Victim||2::Victim||3::Victim||4:...,http://losangeles.cbslocal.com/2013/01/01/man-...,62.0,35.0,2013,1,CA
478855,2013-01-01,Ohio,Lorain,1776 East 28th Street,1,3,http://www.gunviolencearchive.org/incident/478855,http://chronicle.northcoastnow.com/2013/02/14/...,False,9.0,...,0::Damien Bell||1::Desmen Noble||2::Herman Sea...,NaN,"0::Injured, Unharmed, Arrested||1::Unharmed, A...",0::Subject-Suspect||1::Subject-Suspect||2::Vic...,http://www.morningjournal.com/general-news/201...,56.0,13.0,2013,1,OH
478925,2013-01-05,Colorado,Aurora,16000 block of East Ithaca Place,4,0,http://www.gunviolencearchive.org/incident/478925,http://www.dailydemocrat.com/20130106/aurora-s...,False,6.0,...,0::Stacie Philbrook||1::Christopher Ratliffe||...,NaN,0::Killed||1::Killed||2::Killed||3::Killed,0::Victim||1::Victim||2::Victim||3::Subject-Su...,http://denver.cbslocal.com/2013/01/06/officer-...,40.0,28.0,2013,1,CO
478959,2013-01-07,North Carolina,Greensboro,307 Mourning Dove Terrace,2,2,http://www.gunviolencearchive.org/incident/478959,http://www.journalnow.com/news/local/article_d...,False,6.0,...,0::Danielle Imani Jameison||1::Maurice Eugene ...,3::Family,0::Injured||1::Injured||2::Killed||3::Killed,0::Victim||1::Victim||2::Victim||3::Subject-Su...,http://myfox8.com/2013/01/08/update-mother-sho...,62.0,27.0,2013,1,NC


In [39]:
from src.utils import parse_column

df_participants = parse_column(df, "participant_age", "age").merge(
    parse_column(df, "participant_type", "participant_type", stringed=True),
    on=["original_index", "position"],
    how="inner",
)

df_participants.head()

,original_index,position,age,participant_type
0,461105,0,20,Victim
1,460726,0,20,Victim
2,478855,0,25,Subject-Suspect
3,478855,1,31,Subject-Suspect
4,478855,2,33,Victim


In [40]:
df_participants["state"] = df_participants["original_index"].map(df["state"])
df_victims = df_participants[
    (df_participants["participant_type"].str.strip() == "Victim")
    & (df_participants["age"].between(0, 120))
    & (df_participants["state"].notna())
].copy()
df_victims.head()

,original_index,position,age,participant_type,state
0,461105,0,20,Victim,Pennsylvania
1,460726,0,20,Victim,California
4,478855,2,33,Victim,Ohio
5,478855,3,34,Victim,Ohio
6,478855,4,33,Victim,Ohio


In [41]:
df_victims["age"].describe()

count    109047.000000
mean         29.772291
std          13.953467
min           0.000000
25%          20.000000
50%          26.000000
75%          36.000000
max         101.000000
Name: age, dtype: float64

In [42]:
df_victims["age"].median()

np.float64(26.0)

In [43]:
median_victim_age_by_state = (
    df_victims.groupby("state", as_index=False)
    .agg(median_victim_age=("age", "median"), victim_records=("age", "size"))
    .sort_values("median_victim_age", ascending=False)
)
median_victim_age_by_state["code"] = median_victim_age_by_state["state"].map(
    state_codes
)

plot_usa_heatmap(
    median_victim_age_by_state,
    category_col="median_victim_age",
    color_continuous_scale="Viridis",
    title="Median Victim Age by State",
    hover_data=["median_victim_age", "victim_records"],
)

In [44]:
df_suspects = df_participants[
    (df_participants["participant_type"].str.strip() == "Subject-Suspect")
    & df_participants["age"].between(0, 120)
    & df_participants["state"].notna()
].copy()
df_suspects.head()

,original_index,position,age,participant_type,state
2,478855,0,25,Subject-Suspect,Ohio
3,478855,1,31,Subject-Suspect,Ohio
10,478925,3,33,Subject-Suspect,Colorado
14,478959,3,47,Subject-Suspect,North Carolina
24,479363,5,15,Subject-Suspect,New Mexico


In [45]:
df_suspects["age"].describe()

count    112622.000000
mean         29.158069
std          12.200828
min           0.000000
25%          20.000000
50%          26.000000
75%          35.000000
max          98.000000
Name: age, dtype: float64

In [46]:
df_suspects["age"].median()

np.float64(26.0)

In [47]:
median_suspect_age_by_state = (
    df_suspects.groupby("state", as_index=False)
    .agg(median_suspect_age=("age", "median"), suspect_records=("age", "size"))
    .sort_values("median_suspect_age", ascending=False)
)
median_suspect_age_by_state["code"] = median_suspect_age_by_state["state"].map(
    state_codes
)

plot_usa_heatmap(
    median_suspect_age_by_state,
    category_col="median_suspect_age",
    color_continuous_scale="Viridis",
    title="Median Subject-Suspect Age by State",
    hover_data=["median_suspect_age", "suspect_records"],
)